# Kibble-Zurek Mechanism — 3D Ising Model

The Kibble-Zurek mechanism (KZM) predicts that cooling through a phase transition at finite rate τ_Q produces topological defects with density:

  ρ ~ τ_Q^(-κ)   where κ = 2νz / (1 + νz)

For 3D Ising with ν=0.6301 (correlation length exponent) and z=2 (Metropolis dynamic exponent, Model A):

  κ_theory = 2 × 0.6301 × 2 / (1 + 0.6301 × 2) = 1.115

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os

data_dir = 'data'
datasets = {}
for fname in ['kz_N20.csv', 'kz_N30.csv']:
    path = os.path.join(data_dir, fname)
    if os.path.exists(path):
        key = fname.replace('.csv', '')
        datasets[key] = pd.read_csv(path)
        print(f"Loaded {key}: {len(datasets[key])} rows")
    else:
        print(f"WARNING: {fname} not found — run: cargo run --release --bin kz -- --n 20 --trials 10 --tau-min 100 --tau-max 500000 --tau-steps 25 --outdir analysis/data")

# KZM theory
nu = 0.6301
z = 2.0
kzm_exponent = 2 * nu * z / (1 + nu * z)
print(f"\nKZM theory exponent κ = {kzm_exponent:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = {'kz_N20': '#3b82f6', 'kz_N30': '#10b981'}
markers = {'kz_N20': 'o', 'kz_N30': 's'}

for key, df in datasets.items():
    n = key.split('N')[1]
    ax.loglog(df['tau_q'], df['rho'], markers[key] + '-',
              color=colors[key], label=f'N={n}', alpha=0.8, markersize=5)

ax.set_xlabel('τ_Q (quench time, sweeps)', fontsize=12)
ax.set_ylabel('ρ (domain wall density)', fontsize=12)
ax.set_title('Kibble-Zurek: defect density vs quench rate', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(data_dir, 'kz_raw.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
results = []
skip_frac = 0.2  # skip first 20% (short tau_q may be in non-power-law regime)

for key, df in datasets.items():
    n = key.split('N')[1]
    skip = max(1, int(len(df) * skip_frac))
    df_fit = df.iloc[skip:].copy()
    # Only fit rows where rho > 0
    df_fit = df_fit[df_fit['rho'] > 0]
    if len(df_fit) < 5:
        print(f"{key}: not enough non-zero rho points for fit")
        continue
    log_tau = np.log(df_fit['tau_q'].values)
    log_rho = np.log(df_fit['rho'].values)
    slope, intercept, r, p_val, se = stats.linregress(log_tau, log_rho)
    measured_exp = -slope
    pct_err = abs(measured_exp - kzm_exponent) / kzm_exponent * 100
    results.append({
        'dataset': key, 'N': n,
        'kappa_measured': measured_exp,
        'kappa_theory': kzm_exponent,
        'pct_error': pct_err,
        'R2': r**2,
        'se': se
    })
    print(f"N={n}: κ = {measured_exp:.4f} ± {se:.4f}  (theory: {kzm_exponent:.4f}, error: {pct_err:.1f}%, R²={r**2:.4f})")

if results:
    df_results = pd.DataFrame(results)
    print("\nSummary:")
    print(df_results[['dataset', 'kappa_measured', 'kappa_theory', 'pct_error', 'R2']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for key, df in datasets.items():
    n = key.split('N')[1]
    ax.loglog(df['tau_q'], df['rho'], markers[key],
              color=colors[key], label=f'N={n} data', alpha=0.6, markersize=5)

# Theory line scaled to data midpoint
if datasets:
    ref_df = list(datasets.values())[0]
    ref_df_pos = ref_df[ref_df['rho'] > 0]
    if len(ref_df_pos) > 0:
        mid_idx = len(ref_df_pos) // 2
        tau_mid = ref_df_pos.iloc[mid_idx]['tau_q']
        rho_mid = ref_df_pos.iloc[mid_idx]['rho']
        A = rho_mid * tau_mid**kzm_exponent
        tau_ref = np.logspace(np.log10(ref_df_pos['tau_q'].min()),
                              np.log10(ref_df_pos['tau_q'].max()), 100)
        rho_theory = A * tau_ref**(-kzm_exponent)
        ax.loglog(tau_ref, rho_theory, 'k--', linewidth=2,
                  label=f'KZM theory (κ={kzm_exponent:.3f})')

ax.set_xlabel('τ_Q (quench time, sweeps)', fontsize=12)
ax.set_ylabel('ρ (domain wall density)', fontsize=12)
ax.set_title('Kibble-Zurek: power-law fit vs theory', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(data_dir, 'kz_fit.png'), dpi=150, bbox_inches='tight')
plt.show()

## Notes

- κ_theory = 1.115 assumes Metropolis (Model A) dynamics with z=2
- For N=10 (smoke test), rho may be all zeros — small lattices fully order even at fast quench rates
- Finite-size effects suppress defect density for N < correlation length at freeze-out
- For publication-quality results: N=30, 10+ trials, tau_max=10^6